# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

## Preparation: load, audit, and format the data

We begin with the raw dataset because this lab explicitly asks us to use `marketing_customer_analysis.csv`.

The preparation is conservative:

1. Inspect the original dimensions, column names, missing counts, and relevant categorical values.
2. Convert column names to lowercase `snake_case`.
3. Remove only accidental exported-index columns such as `Unnamed: 0`.
4. Standardize only categories actually observed in the data.
5. Convert only the fields required by the exercises to numeric or date types.

We do **not** globally fill missing values. Each analysis drops rows only when one of its required fields is missing, so missingness remains visible and the cleaning does not silently invent data.


In [6]:
import pandas as pd

data_url = (
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/"
    "marketing_customer_analysis.csv"
)
raw_df = pd.read_csv(data_url)

print("Raw shape:", raw_df.shape)
print("Raw columns:")
print(raw_df.columns.tolist())

missing_raw = raw_df.isna().sum()
display(missing_raw[missing_raw > 0].rename("missing_count"))
display(raw_df.head())


Raw shape: (10910, 26)
Raw columns:
['Unnamed: 0', 'Customer', 'State', 'Customer Lifetime Value', 'Response', 'Coverage', 'Education', 'Effective To Date', 'EmploymentStatus', 'Gender', 'Income', 'Location Code', 'Marital Status', 'Monthly Premium Auto', 'Months Since Last Claim', 'Months Since Policy Inception', 'Number of Open Complaints', 'Number of Policies', 'Policy Type', 'Policy', 'Renew Offer Type', 'Sales Channel', 'Total Claim Amount', 'Vehicle Class', 'Vehicle Size', 'Vehicle Type']


,missing_count
State,631
Response,631
Months Since Last Claim,633
Number of Open Complaints,633
Vehicle Class,622
Vehicle Size,622
Vehicle Type,5482


,Unnamed: 0,Customer,State,Customer Lifetime Value,Response,Coverage,Education,Effective To Date,EmploymentStatus,Gender,...,Number of Open Complaints,Number of Policies,Policy Type,Policy,Renew Offer Type,Sales Channel,Total Claim Amount,Vehicle Class,Vehicle Size,Vehicle Type
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2/18/11,Employed,M,...,0.0,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,NaN
1,1,KX64629,California,2228.525238,No,Basic,College,1/18/11,Unemployed,F,...,0.0,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,NaN
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2/10/11,Employed,M,...,0.0,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,1/17/11,Medical Leave,F,...,NaN,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,NaN


In [7]:
df = raw_df.copy()
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# Remove CSV index artifacts, not business data.
index_artifacts = [column for column in df if column.startswith("unnamed")]
df = df.drop(columns=index_artifacts)

audit_columns = [
    "response", "gender", "state", "education",
    "policy_type", "sales_channel"
]
for column in audit_columns:
    if column in df.columns:
        observed = sorted(df[column].dropna().astype(str).unique())
        print(f"{column}: {observed}")


response: ['No', 'Yes']
gender: ['F', 'M']
state: ['Arizona', 'California', 'Nevada', 'Oregon', 'Washington']
education: ['Bachelor', 'College', 'Doctor', 'High School or Below', 'Master']
policy_type: ['Corporate Auto', 'Personal Auto', 'Special Auto']
sales_channel: ['Agent', 'Branch', 'Call Center', 'Web']


The helper below applies only mappings whose source values were observed in the audit. This documents exactly which corrections occurred. Whitespace is stripped from categorical fields first because visually identical labels with hidden spaces should form one group.


In [8]:
def replace_observed(frame, column, candidate_mapping):
    observed = set(frame[column].dropna().unique())
    applied = {
        old: new for old, new in candidate_mapping.items()
        if old in observed
    }
    frame[column] = frame[column].replace(applied)
    print(f"{column} replacements applied: {applied}")


text_columns = df.select_dtypes(include="object").columns
df[text_columns] = df[text_columns].apply(lambda series: series.str.strip())

candidate_rules = {
    "response": {"yes": "Yes", "YES": "Yes", "no": "No", "NO": "No"},
    "gender": {
        "Femal": "F", "female": "F", "Female": "F",
        "Male": "M", "male": "M"
    },
    "state": {"AZ": "Arizona", "Cali": "California", "WA": "Washington"},
    "education": {"Bachelors": "Bachelor"}
}

for column, mapping in candidate_rules.items():
    if column in df.columns:
        replace_observed(df, column, mapping)

numeric_columns = [
    "customer_lifetime_value", "monthly_premium_auto",
    "total_claim_amount", "number_of_policies"
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["effective_to_date"] = pd.to_datetime(
    df["effective_to_date"], errors="coerce"
)

required_columns = {
    "customer", "state", "response", "policy_type", "gender",
    "education", "sales_channel", "customer_lifetime_value",
    "monthly_premium_auto", "total_claim_amount", "effective_to_date"
}
missing_columns = required_columns.difference(df.columns)
assert not missing_columns, f"Required columns missing: {sorted(missing_columns)}"

print("Prepared shape:", df.shape)
display(df.dtypes)


response replacements applied: {}
gender replacements applied: {}
state replacements applied: {}
education replacements applied: {}
Prepared shape: (10910, 25)


/tmp/ipykernel_1519/3788735211.py:35: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["effective_to_date"] = pd.to_datetime(


,0
customer,object
state,object
customer_lifetime_value,float64
response,object
coverage,object
education,object
effective_to_date,datetime64[ns]
employmentstatus,object
gender,object
income,int64


## 1. Customers with low claims who responded “Yes”

Boolean filtering requires both conditions to be true:

- `total_claim_amount < 1000`
- `response == "Yes"`

In pandas, each condition is enclosed in parentheses and combined with `&`, meaning logical **AND**. `.copy()` creates an independent DataFrame and prevents accidental modification of the original.


In [9]:
low_claim_responders = df.loc[
    (df["total_claim_amount"] < 1000)
    & (df["response"] == "Yes")
].copy()

print("Customers meeting both conditions:", len(low_claim_responders))
display(low_claim_responders.head())
display(low_claim_responders["total_claim_amount"].describe())

assert low_claim_responders["total_claim_amount"].lt(1000).all()
assert low_claim_responders["response"].eq("Yes").all()


Customers meeting both conditions: 1399


,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,income,...,number_of_open_complaints,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type
3,XL78013,Oregon,22332.439460,Yes,Extended,College,2011-01-11,Employed,M,49078,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
8,FM55990,California,5989.773931,Yes,Premium,College,2011-01-19,Employed,M,66839,...,0.0,1,Personal Auto,Personal L1,Offer2,Branch,739.200000,Sports Car,Medsize,NaN
15,CW49887,California,4626.801093,Yes,Basic,Master,2011-01-16,Employed,F,79487,...,0.0,1,Special Auto,Special L1,Offer2,Branch,547.200000,SUV,Medsize,NaN
19,NJ54277,California,3746.751625,Yes,Extended,College,2011-02-26,Employed,F,41479,...,1.0,1,Personal Auto,Personal L2,Offer2,Call Center,19.575683,Two-Door Car,Large,A
27,MQ68407,Oregon,4376.363592,Yes,Premium,Bachelor,2011-02-28,Employed,F,63774,...,0.0,1,Personal Auto,Personal L3,Offer2,Agent,60.036683,Four-Door Car,Medsize,NaN


,total_claim_amount
count,1399.000000
mean,412.134087
std,180.460971
min,7.345946
25%,312.000000
50%,398.502948
75%,528.200860
max,960.115399


**Interpretation:** The assertions confirm that every returned row satisfies both requirements. The result excludes missing claim amounts automatically because a missing value cannot evaluate as being below $1,000.


## 2. Responding customers by policy type and gender

We first select customers who responded “Yes”, then group by `policy_type` and `gender`. Named aggregations calculate:

- customer count,
- average monthly premium,
- average customer lifetime value (CLV),
- average total claim amount.

To compare segments, we also calculate two descriptive proxies:

- `claim_to_clv_ratio`: average claim divided by average CLV; lower values indicate less claim cost relative to estimated customer value.
- `value_minus_claim_proxy`: average CLV minus average claim; higher values are more favorable.

These are comparison tools, **not actual profit**, because operating expenses and the timing of revenue and claims are unavailable.


In [10]:
responders = df.loc[df["response"] == "Yes"].dropna(
    subset=[
        "policy_type", "gender", "monthly_premium_auto",
        "customer_lifetime_value", "total_claim_amount"
    ]
)

segment_summary = (
    responders
    .groupby(["policy_type", "gender"], as_index=False)
    .agg(
        customers=("customer", "nunique"),
        average_monthly_premium=("monthly_premium_auto", "mean"),
        average_clv=("customer_lifetime_value", "mean"),
        average_claim=("total_claim_amount", "mean")
    )
)

segment_summary["claim_to_clv_ratio"] = (
    segment_summary["average_claim"] / segment_summary["average_clv"]
)
segment_summary["value_minus_claim_proxy"] = (
    segment_summary["average_clv"] - segment_summary["average_claim"]
)

segment_summary = segment_summary.sort_values(
    ["value_minus_claim_proxy", "claim_to_clv_ratio"],
    ascending=[False, True]
).reset_index(drop=True)

display(segment_summary.round(2))

best_proxy = segment_summary.iloc[0]
lowest_risk = segment_summary.loc[segment_summary["claim_to_clv_ratio"].idxmin()]

print(
    "Highest value-minus-claim proxy: "
    f"{best_proxy['policy_type']} / {best_proxy['gender']}"
)
print(
    "Lowest claim-to-CLV ratio: "
    f"{lowest_risk['policy_type']} / {lowest_risk['gender']}"
)
print("Reminder: these indicate relative attractiveness, not accounting profit.")


,policy_type,gender,customers,average_monthly_premium,average_clv,average_claim,claim_to_clv_ratio,value_minus_claim_proxy
0,Personal Auto,F,479,99.00,8339.79,452.97,0.05,7886.83
1,Special Auto,M,30,86.34,8247.09,429.53,0.05,7817.56
2,Corporate Auto,M,139,92.19,7944.47,408.58,0.05,7535.88
3,Corporate Auto,F,149,94.30,7712.63,433.74,0.06,7278.89
4,Special Auto,F,32,92.31,7691.58,453.28,0.06,7238.30
5,Personal Auto,M,479,91.09,7448.38,457.01,0.06,6991.37


Highest value-minus-claim proxy: Personal Auto / F
Lowest claim-to-CLV ratio: Corporate Auto / M
Reminder: these indicate relative attractiveness, not accounting profit.


**Interpretation:** Favorable segments combine higher CLV and premiums with lower claims, but group size matters. A result based on very few customers is less stable, which is why the table includes `customers`. Actual profitability would require earned premium, acquisition costs, servicing costs, claim timing, and policy duration.


## 3. States with more than 500 customers

Because the question asks for customers rather than rows, we use `nunique()` on the customer identifier within each state. This remains correct if a customer appears in more than one record. We then filter the aggregated Series to counts greater than 500.


In [11]:
customers_by_state = (
    df.dropna(subset=["state", "customer"])
    .groupby("state")["customer"]
    .nunique()
    .sort_values(ascending=False)
    .rename("number_of_customers")
)

states_over_500 = customers_by_state.loc[customers_by_state > 500]

display(customers_by_state)
print("States with more than 500 customers:")
display(states_over_500)

assert states_over_500.gt(500).all()


,number_of_customers
state,
California,3150
Oregon,2601
Arizona,1703
Nevada,882
Washington,798


States with more than 500 customers:


,number_of_customers
state,
California,3150
Oregon,2601
Arizona,1703
Nevada,882
Washington,798


**Interpretation:** The full Series provides context, while the second contains only qualifying states. The comparison is strictly `> 500`, so a state with exactly 500 customers would not be included.


## 4. CLV statistics by education and gender

Grouping by both categorical columns creates one row per education–gender segment. Named aggregations return the maximum, minimum, median, and group size. The median represents the middle observation and is less affected by extreme CLV values than the mean.


In [12]:
clv_by_education_gender = (
    df.dropna(subset=["education", "gender", "customer_lifetime_value"])
    .groupby(["education", "gender"], as_index=False)
    .agg(
        maximum_clv=("customer_lifetime_value", "max"),
        minimum_clv=("customer_lifetime_value", "min"),
        median_clv=("customer_lifetime_value", "median"),
        customers=("customer", "nunique")
    )
    .sort_values("median_clv", ascending=False)
    .reset_index(drop=True)
)

display(clv_by_education_gender.round(2))

highest_median = clv_by_education_gender.iloc[0]
lowest_median = clv_by_education_gender.iloc[-1]
print(
    "Highest median CLV: "
    f"{highest_median['education']} / {highest_median['gender']} "
    f"({highest_median['median_clv']:,.2f})"
)
print(
    "Lowest median CLV: "
    f"{lowest_median['education']} / {lowest_median['gender']} "
    f"({lowest_median['median_clv']:,.2f})"
)


,education,gender,maximum_clv,minimum_clv,median_clv,customers
0,High School or Below,M,83325.38,1940.98,6286.73,1301
1,High School or Below,F,55277.45,2144.92,6039.55,1321
2,College,M,61134.68,1918.12,6005.85,1329
3,Master,F,51016.07,2417.78,5729.86,393
4,Bachelor,F,73225.96,1904.00,5640.51,1423
5,College,F,61850.19,1898.68,5623.61,1352
6,Master,M,50568.26,2272.31,5579.10,348
7,Doctor,M,32677.34,2267.60,5577.67,173
8,Bachelor,M,67907.27,1898.01,5548.03,1325
9,Doctor,F,44856.11,2395.57,5332.46,169


Highest median CLV: High School or Below / M (6,286.73)
Lowest median CLV: Doctor / F (5,332.46)


**Conclusion:** The dynamically printed groups identify the highest and lowest typical CLV. Compare `maximum_clv` with `median_clv` to spot upper-tail outliers. Also inspect `customers`: conclusions from small groups should be treated cautiously.


## 5. Bonus — policies sold by state and month

Each row is treated as one policy record, which is the dataset structure assumed by the lab. We derive month names from `effective_to_date`, then use a pivot table with states as rows and months as columns. `aggfunc="size"` counts records and `fill_value=0` represents combinations with no policies.

A categorical month order prevents alphabetical sorting.


In [13]:
month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

policy_data = df.dropna(subset=["state", "effective_to_date"]).copy()
policy_data["month"] = pd.Categorical(
    policy_data["effective_to_date"].dt.month_name(),
    categories=month_order,
    ordered=True
)

policies_by_state_month = policy_data.pivot_table(
    index="state",
    columns="month",
    aggfunc="size",
    fill_value=0,
    observed=False
)

display(policies_by_state_month)
print("Policies represented in pivot:", int(policies_by_state_month.to_numpy().sum()))
print("Dated policy rows:", len(policy_data))
assert policies_by_state_month.to_numpy().sum() == len(policy_data)


month,January,February,March,April,May,June,July,August,September,October,November,December
state,,,,,,,,,,,,
Arizona,1008,929,0,0,0,0,0,0,0,0,0,0
California,1918,1634,0,0,0,0,0,0,0,0,0,0
Nevada,551,442,0,0,0,0,0,0,0,0,0,0
Oregon,1565,1344,0,0,0,0,0,0,0,0,0,0
Washington,463,425,0,0,0,0,0,0,0,0,0,0


Policies represented in pivot: 10279
Dated policy rows: 10279


## 6. Bonus — monthly policy sales for the top three states

Summing across the month columns gives total policy records per state. `nlargest(3)` selects the top three totals, and `.loc[...]` uses those state labels to filter the original pivot table.


In [14]:
state_policy_totals = policies_by_state_month.sum(axis=1)
top_three_states = state_policy_totals.nlargest(3).index

top_three_state_monthly_sales = policies_by_state_month.loc[top_three_states].copy()
top_three_state_monthly_sales["Total"] = top_three_state_monthly_sales.sum(axis=1)

display(top_three_state_monthly_sales)
print("Top three states:", top_three_states.tolist())


month,January,February,March,April,May,June,July,August,September,October,November,December,Total
state,,,,,,,,,,,,,
California,1918,1634,0,0,0,0,0,0,0,0,0,0,3552
Oregon,1565,1344,0,0,0,0,0,0,0,0,0,0,2909
Arizona,1008,929,0,0,0,0,0,0,0,0,0,0,1937


Top three states: ['California', 'Oregon', 'Arizona']


**Interpretation:** The `Total` column confirms why these states were selected. Reading across a row shows seasonality within one state; reading down a month compares the three leading states during the same period.


## 7. Bonus — response rate by sales channel using `melt`

Response rate is the percentage of customers in a channel whose response is `Yes`, not simply the number of “Yes” responses. We create a Boolean indicator, average it within each channel, and multiply by 100.

The intermediate result is a one-row **wide** table with channels as columns. `melt()` unpivots it into a tidy **long** table with one channel per row, as requested.


In [15]:
channel_data = df.dropna(subset=["sales_channel", "response"]).copy()
channel_data["responded_yes"] = channel_data["response"].eq("Yes")

response_rate_wide = (
    channel_data.groupby("sales_channel")["responded_yes"]
    .mean()
    .mul(100)
    .to_frame()
    .T
)
response_rate_wide.index = ["response_rate_percent"]

response_rate_long = (
    response_rate_wide
    .reset_index(names="metric")
    .melt(
        id_vars="metric",
        var_name="sales_channel",
        value_name="response_rate_percent"
    )
    .sort_values("response_rate_percent", ascending=False)
    .reset_index(drop=True)
)

display(response_rate_wide.round(2))
display(response_rate_long.round(2))

top_response_channel = response_rate_long.iloc[0]
print(
    f"Highest response rate: {top_response_channel['sales_channel']} "
    f"({top_response_channel['response_rate_percent']:.2f}%)"
)


sales_channel,Agent,Branch,Call Center,Web
response_rate_percent,19.07,11.38,10.98,11.71


,metric,sales_channel,response_rate_percent
0,response_rate_percent,Agent,19.07
1,response_rate_percent,Web,11.71
2,response_rate_percent,Branch,11.38
3,response_rate_percent,Call Center,10.98


Highest response rate: Agent (19.07%)


**Interpretation:** The first row of the sorted long table is the strongest observed channel by response rate. Rates make channels with different audience sizes comparable, but the result is descriptive: it does not prove the channel caused the response. Add channel sample sizes and campaign assignment information before making a causal marketing decision.


## Final validation

The closing checks summarize the evidence used throughout the notebook: prepared row count, duplicates, remaining missing values in required analysis columns, and canonical categorical labels. They make unexpected data changes visible.


In [16]:
print("Prepared rows:", len(df))
print("Exact duplicate rows retained in source:", df.duplicated().sum())

analysis_columns = sorted(required_columns)
remaining_missing = df[analysis_columns].isna().sum()
display(remaining_missing[remaining_missing > 0].rename("remaining_missing"))

for column in ["response", "gender", "policy_type", "sales_channel"]:
    print(f"{column}: {sorted(df[column].dropna().astype(str).unique())}")


Prepared rows: 10910
Exact duplicate rows retained in source: 37


,remaining_missing
response,631
state,631


response: ['No', 'Yes']
gender: ['F', 'M']
policy_type: ['Corporate Auto', 'Personal Auto', 'Special Auto']
sales_channel: ['Agent', 'Branch', 'Call Center', 'Web']
